In [0]:
# 1. PARÂMETROS DO AMBIENTE

# DECISÃO: centralizar catalog/schema em variáveis Python, em vez de fazer
# hardcode de "workspace.bronze.tb_x" em cada write. Isso permite trocar de
# ambiente (dev/staging/prod) ou de workspace inteiro alterando 3 linhas,
# e evita o erro clássico de "esqueci de trocar um dos nomes" quando o
# catalog muda. O custo é uma indireção a mais para quem lê o notebook pela
# primeira vez — aceitável dado o ganho de manutenibilidade

catalog = "workspace"
bronze_schema_name = "bronze"
silver_schema_name = "silver"
gold_schema_name = "gold"

bronze_schema = f"{catalog}.{bronze_schema_name}"
silver_schema = f"{catalog}.{silver_schema_name}"
gold_schema = f"{catalog}.{gold_schema_name}"

# Caminho do Volume criado para os inputs
landing_path = f"/Volumes/{catalog}/default/inputs"

print(f"catalog: {catalog}")
print(f"bronze_schema: {bronze_schema}")
print(f"silver_schema: {silver_schema}")
print(f"gold_schema: {gold_schema}")
print(f"landing_path: {landing_path}")

catalog: workspace
bronze_schema: workspace.bronze
silver_schema: workspace.silver
gold_schema: workspace.gold
landing_path: /Volumes/workspace/default/inputs


In [0]:

# 2. PROVISIONAMENTO DOS SCHEMAS (DDL NO UNITY CATALOG)

# DECISÃO: `CREATE SCHEMA IF NOT EXISTS` em vez de assumir que o ambiente já
# está provisionado. Como este notebookvai rodar o Job em agendamento
# (requisito do projeto), ele precisa ser idempotente — se falhar na 2ª
# execução por "schema already exists", o Workflow inteiro quebra sem motivo
# de negócio real, só por falta de robustez do script


spark.sql(f"CREATE SCHEMA IF NOT EXISTS {bronze_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {silver_schema}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {gold_schema}")

print("Schemas bronze, silver e gold provisionados com sucesso.")

Schemas bronze, silver e gold provisionados com sucesso.


In [0]:
# 3. Entrada de Parâmetros via Widgets

# DECISÃO: widgets de texto (não date pickers) porque o Databricks Jobs
# injeta parâmetros como string no formato exigido pela API (MM-DD-AAAA) via
# `base_parameters`. Deixamos o valor default vazio de propósito: assim o
# notebook decide sozinho (célula 6) qual janela consultar quando ninguém
# passar parâmetro, facilitando o comportamento de "modo produção" vs. "modo teste manual"
# no mesmo código, sem precisar de dois notebooks


dbutils.widgets.text("data_inicio", "", "Data Início PTAX (MM-DD-AAAA)")
dbutils.widgets.text("data_fim", "", "Data Fim PTAX (MM-DD-AAAA)")

data_inicio_widget = dbutils.widgets.get("data_inicio").strip()
data_fim_widget = dbutils.widgets.get("data_fim").strip()

print(f"Parâmetros informados -> Início: '{data_inicio_widget}' | Fim: '{data_fim_widget}'")

Parâmetros informados -> Início: '' | Fim: ''


In [0]:
# 4. MAPEAMENTO E LEITURA BRUTA DOS ARQUIVOS 

# DECISÃO: `inferSchema=False` é deliberado, não just um detalhe técnico.
# Se deixássemos o Spark inferir tipos aqui, colunas com Column Shift (ex.:
# texto solto em vote_average) já seriam silenciosamente convertidas/
# descartadas na Bronze — perdendo rastreabilidade do dado original "sujo"
# exigida pela arquitetura Medalhão. Ao manter tudo como string, empurramos
# a decisão de tipagem (com suas regras de negócio) para a Silver, onde ela
# é auditável e documentada.
# `multiLine=True` + `quote/escape=` evitam que sinopses com quebras de
# linha ou vírgulas internas fragmentem o CSV em colunas erradas. Sem isso,
# o dataset inteiro desalinharia silenciosamente a partir da primeira ocorrência



# 1. Caminhos absolutos validados no Volume
path_movies_info = f"{landing_path}/movies_info_TMDB_IMDB.csv"
path_movies_financials = f"{landing_path}/movies_financials_IMDB_TMDB.csv"
path_movies_metrics = f"{landing_path}/movies_metrics_IMDB_TMDB.csv"
path_credits_and_tags = f"{landing_path}/credits_and_tags_IMDB_TMDB.csv"
path_movies_reviews = f"{landing_path}/movies_reviews.csv"

# 2. Leitura pura dos CSVs no Spark Engine
df_movies_info_raw = spark.read.csv(path_movies_info, header=True, inferSchema=False, multiLine=True, quote='"', escape='"')
df_movies_financials_raw = spark.read.csv(path_movies_financials, header=True, inferSchema=False, multiLine=True, quote='"', escape='"')
df_movies_metrics_raw = spark.read.csv(path_movies_metrics, header=True, inferSchema=False, multiLine=True, quote='"', escape='"')
df_credits_and_tags_raw = spark.read.csv(path_credits_and_tags, header=True, inferSchema=False, multiLine=True, quote='"', escape='"')
df_movies_reviews_raw = spark.read.csv(path_movies_reviews, header=True, inferSchema=False, multiLine=True, quote='"', escape='"')

print("Leitura dos 5 arquivos CSV concluída com sucesso no Spark!")

Leitura dos 5 arquivos CSV concluída com sucesso no Spark!


In [0]:
# 5. PERSISTÊNCIA NA CAMADA BRONZE (APPEND MODE + INGESTION DATETIME)

# DECISÃO: Uso do`mode("append")` conforme exigido no escopo do projeto.
# Bronze é a camada de histórico bruto — cada execução do Job deve ACRESCENTAR
# uma nova leva de dados com seu próprio `ingestion_datetime`, permitindo à
# Silver escolher qual versão é a mais recente (célula 2 do Bronze_to_Silver).
# Se usássemos "overwrite" aqui, perderíamos a capacidade de auditar/reprocessar
# cargas anteriores — um dos motivos de existir a arquitetura Medalhão
#importe do módulo para manipulação de datas
from pyspark.sql.functions import current_timestamp

# Gravação da tabela de informações cadastrais dos filmes
df_movies_info_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_info")

# Gravação da tabela financeira
df_movies_financials_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_financials")

# Gravação da tabela de métricas
df_movies_metrics_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_metrics")

# Gravação da tabela de créditos e tags
df_credits_and_tags_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_credits_and_tags")

# Gravação da tabela de reviews de usuários
df_movies_reviews_raw \
    .withColumn("ingestion_datetime", current_timestamp()) \
    .write.format("delta").mode("append") \
    .saveAsTable(f"{bronze_schema}.tb_movies_reviews")

print("Todas as 5 tabelas Delta dos arquivos CSV foram gravadas em modo Append.")

Todas as 5 tabelas Delta dos arquivos CSV foram gravadas em modo Append.


In [0]:
# 6. INGESTÃO DE API (BANCO CENTRAL DO BRASIL - PTAX)

# DECISÃO: fallback de 7 dias corridos quando os widgets vêm vazios, em vez
# de abortar a execução. Justificativa: a API do BCB não retorna cotação em
# fins de semana/feriados, então uma janela de exatamente "hoje" corre risco
# real de vir vazia; 7 dias garante ao menos 1 dia útil disponível


import requests
from datetime import datetime, timedelta
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

# 1. Definição do intervalo de datas (MM-DD-AAAA)
data_hoje = datetime.today()

if data_fim_widget:
    data_fim_req = data_fim_widget
else:
    data_fim_req = data_hoje.strftime("%m-%d-%Y")

if data_inicio_widget:
    data_inicio_req = data_inicio_widget
else:
    data_inicio_req = (data_hoje - timedelta(days=7)).strftime("%m-%d-%Y")

print(f"Executando chamada à API do BCB: {data_inicio_req} até {data_fim_req}")

# 2. Montagem do endpoint  PTAX OData
endpoint_ptax = (
    f"https://olinda.bcb.gov.br/olinda/servico/PTAX/versao/v1/odata/"
    f"CotacaoDolarPeriodo(dataInicial=@dataInicial,dataFinalCotacao=@dataFinalCotacao)?"
    f"@dataInicial='{data_inicio_req}'&@dataFinalCotacao='{data_fim_req}'&"
    f"$select=dataHoraCotacao,cotacaoCompra&$format=json"
)

# 3. Requisição HTTP com timeout explícito
response = requests.get(endpoint_ptax, timeout=30)

if response.status_code != 200:
    raise RuntimeError(f"Erro na requisição à API PTAX ({response.status_code}): {response.text}")

cotacoes_json = response.json().get("value", [])

if not cotacoes_json:
    print(f"Atenção: Nenhuma cotação foi retornada pela API para o período informado.")
else:
    # 4. Tipagem do payload retornado
    schema_cotacao = StructType([
        StructField("dataHoraCotacao", StringType(), True),
        StructField("cotacaoCompra", DoubleType(), True)
    ])
    
    # 5. Criação do DataFrame e persistência Delta com ingestion_datetime
    df_cotacao_raw = spark.createDataFrame(cotacoes_json, schema=schema_cotacao)
    
    df_cotacao_raw \
        .withColumn("ingestion_datetime", current_timestamp()) \
        .write.format("delta").mode("append") \
        .saveAsTable(f"{bronze_schema}.tb_cotacao_dolar")
    
    print(f"✔ Tabela {bronze_schema}.tb_cotacao_dolar gravada com sucesso ({df_cotacao_raw.count()} registros).")

Executando chamada à API do BCB: 09-14-2026 até 09-21-2026
✔ Tabela workspace.bronze.tb_cotacao_dolar gravada com sucesso (5 registros).


In [0]:
# 7. VALIDAÇÃO DAS TABELAS GRAVADAS (SMOKE TEST)
# DECISÃO: contagem de linhas + amostra visual em vez de um teste de schema
# completo. Para o escopo do projeto, um smoke test simples já detecta
# os dois erros mais prováveis num Job agendado (tabela vazia / tabela não
# criada) sem o custo de manter testes de schema completos. Um projeto real
# em produção evoluiria isso para expectations (ex.: Great Expectations) ou
# `dbutils.notebook.exit()` com status de falha para o orquestrador reagir


tabelas_bronze = [
    "tb_movies_info",
    "tb_movies_financials",
    "tb_movies_metrics",
    "tb_credits_and_tags",
    "tb_movies_reviews",
    "tb_cotacao_dolar"
]

print("=" * 65)
print(f"VERIFICAÇÃO DE TABELAS CRIADAS NO CATALOG '{catalog}':")
print("=" * 65)

for tab in tabelas_bronze:
    nome_completo = f"{bronze_schema}.{tab}"
    try:
        contagem = spark.table(nome_completo).count()
        print(f"{nome_completo:<38} | Linhas: {contagem}")
    except Exception as e:
        print(f"Falha ao consultar {nome_completo}: {e}")

print("=" * 65)

# Exibe uma amostra da tabela da API para checagem visual imediata
display(spark.table(f"{bronze_schema}.tb_cotacao_dolar").limit(5))

VERIFICAÇÃO DE TABELAS CRIADAS NO CATALOG 'workspace':
workspace.bronze.tb_movies_info        | Linhas: 746172
workspace.bronze.tb_movies_financials  | Linhas: 743155
workspace.bronze.tb_movies_metrics     | Linhas: 721504
workspace.bronze.tb_credits_and_tags   | Linhas: 739256
workspace.bronze.tb_movies_reviews     | Linhas: 226884
workspace.bronze.tb_cotacao_dolar      | Linhas: 35


dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-14 13:10:08.144425,5.169,2026-09-21T14:19:17.337Z
2026-09-15 13:09:19.199664,5.1484,2026-09-21T14:19:17.337Z
2026-09-16 13:05:30.35873,5.152,2026-09-21T14:19:17.337Z
2026-09-17 13:03:21.858212,5.1515,2026-09-21T14:19:17.337Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T14:19:17.337Z


In [0]:
%sql
/*  DECISÃO: inspeção manual via SQL puro no fim do notebook, separada do
smoke test em Python (célula 7). Mantemos os dois porque servem públicos
 diferentes: a célula 7 é para o Job (log automatizado, decide sucesso/
 falha), esta é para quem está revisando o notebook manualmente e quer ver
o dado "cru" como ele chegou, sem qualquer transformação da Silver. */


SELECT * FROM bronze.tb_movies_info LIMIT 5;
SELECT * FROM bronze.tb_movies_financials LIMIT 5;
SELECT * FROM bronze.tb_movies_metrics LIMIT 5;
SELECT * FROM bronze.tb_credits_and_tags LIMIT 5;
SELECT * FROM bronze.tb_movies_reviews LIMIT 5;
SELECT * FROM bronze.tb_cotacao_dolar LIMIT 5;

dataHoraCotacao,cotacaoCompra,ingestion_datetime
2026-09-14 13:10:08.144425,5.169,2026-09-21T14:19:17.337Z
2026-09-15 13:09:19.199664,5.1484,2026-09-21T14:19:17.337Z
2026-09-16 13:05:30.35873,5.152,2026-09-21T14:19:17.337Z
2026-09-17 13:03:21.858212,5.1515,2026-09-21T14:19:17.337Z
2026-09-18 13:03:34.742036,5.1569,2026-09-21T14:19:17.337Z
